In [ ]:
import os
os.environ["HUGGINGFACE_API"] = ""
os.environ["GIT_TOKEN"] = ""

In [ ]:
## Using google colab
# import os
# from google.colab import userdata

# # Lấy các token từ Colab Secrets (biểu tượng 🔑 bên trái)
# try:
#     os.environ["HUGGINGFACE_API"] = userdata.get('HF_TOKEN')
#     os.environ["GIT_TOKEN"] = userdata.get('GIT_TOKEN')
#     os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
# except userdata.SecretNotFoundError:
#     print(userdata.SecretNotFoundError)

In [ ]:
import os
from huggingface_hub import login


huggingface_api = os.environ["HUGGINGFACE_API"]
git_token = os.environ["GIT_TOKEN"]

if huggingface_api is None:
    raise RuntimeError("❌ Missing HUGGINGFACE_API in .env")

login(token=huggingface_api)


In [ ]:
!git clone https://{git_token}@github.com/BGKhanh/Reasoning-Techniques-on-LLM.git

In [ ]:
%cd /Reasoning-Techniques-on-LLM

# GEPA Optimization for Vietnamese SSA

Tối ưu prompt **Chain-of-Thought** cho task Structured Sentiment Analysis tiếng Việt
bằng `dspy.GEPA` (Genetic-Pareto reflective prompt optimizer, [arxiv:2507.19457](https://arxiv.org/abs/2507.19457)).

**Pipeline:**
1. Load 3 splits (train/dev/test) từ `data/vitoed_new/`
2. Định nghĩa `dspy.Signature` với system prompt làm docstring (zero-shot)
3. Wrap thành `dspy.ChainOfThought` module
4. Baseline eval trên valset
5. `dspy.GEPA.compile()` với feedback metric (decompose 6 SemEval-2022 F1)
6. So sánh trước/sau optimization trên test set

**Lưu ý:** Notebook này chỉ tối ưu cho **Chain-of-Thought**.
Các technique khác (Few-shot, Re-reading, Plan-and-Solve) giữ nguyên ở `src/prompt_templates/`
và chạy qua `lm-eval` riêng.

In [ ]:
!pip install -q dspy underthesea

In [ ]:
# Uncomment nếu chưa cài (đã có trong requirements.txt: dspy>=3.1.3, underthesea, transformers, ...)
# !pip install -q "dspy>=3.1.3" gepa underthesea

import os
import sys
import json
import random
from pathlib import Path
from typing import Any, Dict, List, Optional

import dspy

# Resolve project root robustly: walk up từ cwd để tìm dir chứa `src/prompt_templates`.
# Robust với mọi cwd: notebook/ (local), project root (CI), lm-evaluation-harness/ (vastai cd).
def _find_project_root(start: Path, max_up: int = 5) -> Path:
    cur = start.resolve()
    for _ in range(max_up):
        if (cur / "src" / "prompt_templates").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return start  # fallback — import sẽ FAIL với message rõ ràng


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Project-local helpers (zero side effects on import)
from src.prompt_templates.shared import SYSTEM_PROMPTS
from src.utils.postprocessing import extract_json_from_response, postprocess_response
from semeval22_structured_sentiment.evaluation.evaluate import (
    convert_opinion_to_tuple,
    sent_tuples_in_list,
    calculate_all_metrics,  # 6 SemEval-2022 F1 — single source of truth
)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"dspy version = {getattr(dspy, '__version__', 'unknown')}")

## ⚙️ Configuration

Single source-of-truth: chỉnh **cell Configuration** (block `TECHNIQUE` / `DATA_DIR` / models) rồi run-all để swap model / budget / dataset size.
Các cell khác đều derive từ block này.

In [ ]:
# ===================================================================
# Technique (notebook này chỉ hỗ trợ Chain-of-Thought)
# ===================================================================
TECHNIQUE = "cot"

# DATA_DIR      = PROJECT_ROOT / "semeval22_structured_sentiment" / "data"
# DATASETS      = ["opener_en", "opener_es", "norec", "multibooked_eu", "multibooked_ca"]
# PERCENT_DATA  = [2, 2, 2, 2, 2]   # None ở vị trí nào → bỏ dataset đó
# TRAINSET_SIZE = 7000
# VALSET_SIZE   = 100
# TESTSET_SIZE  = 1000
# SEED          = 42

# DATA_DIR      = PROJECT_ROOT / "semeval22_structured_sentiment" / "data"
# DATASETS      = ["multibooked_ca"]
# PERCENT_DATA  = [1]   # None ở vị trí nào → bỏ dataset đó
# TRAINSET_SIZE = 500
# VALSET_SIZE   = 50
# TESTSET_SIZE  = None
# SEED          = 42

DATA_DIR = PROJECT_ROOT / "data"
DATASETS = ["vitoed_new"]
PERCENT_DATA = [1]
TRAINSET_SIZE = None
VALSET_SIZE   = 100
TESTSET_SIZE  = None
SEED          = 42

DATASET_LANG: Dict[str, str] = {
    "opener_en":      "en",
    "opener_es":      "es",
    "norec":          "no",
    "multibooked_eu": "eu",
    "multibooked_ca": "ca",
    "vitoed_new":     "vi",
}

# ===================================================================
# Models
# Student: local vLLM (OpenAI-compatible). Tránh LiteLLM → Hugging Face Inference API (401 nếu thiếu key).
# Reflection: model mạnh (GPT-5/Claude Opus/Gemini-2.5-Pro) — chất lượng prompt phụ thuộc nặng vào đây.
# ===================================================================
# from google.colab import userdata
# try:
#     GEMINI_API_KEY = userdata.get('GOOGLE_API_KEY')
# except:
#     GEMINI_API_KEY = os.getenv("GOOGLE_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")

# vLLM: `vllm serve` đăng ký model theo `--model` (mặc định tên = repo HF). LiteLLM cần prefix `openai/`.
VLLM_MODEL_ID = os.getenv("VLLM_MODEL_ID", "google/gemma-3-4b-it")
VLLM_PORT = int(os.getenv("VLLM_PORT", "8000"))
STUDENT_BASE_URL = os.getenv("STUDENT_BASE_URL", f"http://127.0.0.1:{VLLM_PORT}/v1")
STUDENT_API_KEY = os.getenv("STUDENT_API_KEY", "EMPTY")  # vLLM không bắt buộc; LiteLLM vẫn cần chuỗi
STUDENT_MODEL = f"openai/{VLLM_MODEL_ID}"

# REFLECTION_MODEL = "gemini/gemma-4-31b-it"   # WARN: 'gemini/' prefix dành cho Gemini family — đổi thành 'gemini/gemini-2.5-pro' hoặc 'vertex_ai/google/gemma-3-27b-it'
REFLECTION_MODEL = "gemini/gemini-3.1-flash-lite" 
STUDENT_MAX_TOKENS = 8196
REFLECTION_MAX_TOKENS = 64000

# ===================================================================
# GEPA budget
# ===================================================================
REFLECTION_MINIBATCH_SIZE = 100
NUM_THREADS = 4
# Exactly one of max_metric_calls, max_full_evals, auto must be set.
AUTO_BUDGET = "heavy"
MAX_METRIC_CALLS = None
MAX_FULL_EVALS = None
# Prompt lenght punishment
PROMPT_LENGTH_BASELINE = 3000
PROMPT_LENGTH_TARGET   = 4000
PROMPT_LENGTH_HARD_CAP = 5000

# Penalty tối đa trong vùng hợp lệ
MAX_LENGTH_PENALTY = 0.20

# Phạt thêm sau HARD_CAP
MIN_MULTIPLIER = 0.5
OVERFLOW_SCALE = 1500
# ===================================================================
# Output
# ===================================================================
OUTPUT_DIR = PROJECT_ROOT / "results" / "gepa" / TECHNIQUE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GEPA_LOG_DIR = OUTPUT_DIR / "log"
GEPA_LOG_DIR.mkdir(parents=True, exist_ok=True)

import random
random.seed(SEED)
print(f"GEMINI_API_KEY: {'Loaded' if GEMINI_API_KEY else 'Missing'}")

In [ ]:
# from google import genai

# client = genai.Client(api_key=GEMINI_API_KEY)

# for model in client.models.list():
#     print(model.name)

## 📦 Data Pipeline

Convert raw VLSP samples → `dspy.Example` với `inputs = (text, sent_id)`.
Field `opinions` giữ nguyên format SemEval (offsets) để metric tái sử dụng đúng logic
`convert_opinion_to_tuple` từ `semeval22_structured_sentiment/evaluation/evaluate.py`.

In [ ]:
from semeval22_structured_sentiment.evaluation.evaluate import set_tokenizer
from src.utils.postprocessing import clean_gold_data

def _shuffle_take(items: List[dspy.Example], n: Optional[int], seed: int) -> List[dspy.Example]:
    """Deterministic shuffle + take n; n=None → take all."""
    if n is None or n >= len(items):
        return list(items)
    rng = random.Random(seed)
    pool = list(items)
    rng.shuffle(pool)
    return pool[:n]

def load_examples(
    path: Path,
    language: str = "en",
    clean: bool = True,
    verbose: bool = True,
) -> List[dspy.Example]:
    if not path.exists():
        raise FileNotFoundError(f"Data file not found: {path}")
    with path.open("r", encoding="utf-8") as f:
        raw = json.load(f)
    if clean:
        set_tokenizer(language)
        raw, removed_ids, _ = clean_gold_data(raw, language=language, verbose=verbose)
        if removed_ids and verbose:
            print(f"  [{path.stem}] Removed {len(removed_ids)}: {removed_ids}\n")
    return [
        dspy.Example(
            text=str(s["text"]),
            sent_id=str(s["sent_id"]),
            opinions=s.get("opinions", []),
            dataset=path.parent.name,    # ← track nguồn gốc
        ).with_inputs("text")
        for s in raw
    ]


# =========================================================================
# Load + mix nhiều datasets theo PERCENT_DATA
# =========================================================================

def load_split_from_datasets(
    split: str,
    total_size: Optional[int],
    seed: int,
    datasets: List[str]          = DATASETS,
    percent_data: List[Optional[int]] = PERCENT_DATA,
    data_dir: Path               = DATA_DIR,
    clean: bool                  = True,
    verbose: bool                = True,
) -> List[dspy.Example]:
    """Load và mix nhiều datasets theo tỉ lệ PERCENT_DATA.

    Args:
        split        : "train" | "dev" | "test"
        total_size   : Tổng số sample (None = lấy hết)
        seed         : Random seed cho shuffle + sampling
        datasets     : Danh sách tên dataset (song song với percent_data)
        percent_data : Trọng số (None ở vị trí nào → bỏ dataset đó)
        data_dir     : Root chứa các dataset
        clean        : Chạy clean_gold_data
        verbose      : In tiến trình
    """
    # ── Lọc dataset hợp lệ ────────────────────────────────────────
    active = [
        (name, w)
        for name, w in zip(datasets, percent_data)
        if w is not None
    ]
    if not active:
        raise ValueError("PERCENT_DATA cần ít nhất 1 giá trị khác None.")

    total_weight = sum(w for _, w in active)

    if verbose:
        print(f"\n{'='*60}")
        print(f"Loading split='{split}' | total_size={total_size}")
        print(f"Active datasets: {[(n, w) for n, w in active]}")
        print(f"{'='*60}")

    # ── Load từng dataset ──────────────────────────────────────────
    pool: Dict[str, List[dspy.Example]] = {}
    for name, _ in active:
        lang = DATASET_LANG.get(name, "en")
        path = data_dir / name / f"{split}.json"
        pool[name] = load_examples(path, language=lang, clean=clean, verbose=verbose)

    # ── Lấy hết nếu total_size=None ───────────────────────────────
    if total_size is None:
        result = [ex for exs in pool.values() for ex in exs]
        random.Random(seed).shuffle(result)
        return result

    # ── Phân bổ proportional ───────────────────────────────────────
    allocations: Dict[str, int] = {
        name: round(total_size * w / total_weight)
        for name, w in active
    }
    # Sửa sai số làm tròn vào dataset đầu tiên
    diff = total_size - sum(allocations.values())
    allocations[active[0][0]] += diff

    # ── Sample + mix ──────────────────────────────────────────────
    result: List[dspy.Example] = []
    if verbose:
        print(f"\n{'Dataset':<20} {'Available':>10} {'Target':>8} {'Taken':>8}")
        print("-" * 50)

    for name, _ in active:
        target = allocations[name]
        taken  = _shuffle_take(pool[name], target, seed)
        result.extend(taken)
        if verbose:
            flag = "✓" if len(taken) >= target else f"⚠ capped"
            print(f"  {name:<18} {len(pool[name]):>10} {target:>8} {len(taken):>8}  {flag}")

    random.Random(seed).shuffle(result)
    return result


# =========================================================================
# Load splits
# =========================================================================

trainset = load_split_from_datasets("train", TRAINSET_SIZE, SEED, verbose=False)
valset   = load_split_from_datasets("dev",   VALSET_SIZE,   SEED + 1, verbose=False)
testset  = load_split_from_datasets("test",  TESTSET_SIZE,  SEED + 2, verbose=False)


# =========================================================================
# Distribution stats
# =========================================================================

def _polarity_dist(exs: List[dspy.Example]) -> tuple[dict, int]:
    dist: Dict[str, int] = {}
    for ex in exs:
        for op in ex.opinions:
            pol = op.get("Polarity", "Unknown")
            dist[pol] = dist.get(pol, 0) + 1
    return dist, sum(dist.values())

def _dataset_dist(exs: List[dspy.Example]) -> Dict[str, int]:
    dist: Dict[str, int] = {}
    for ex in exs:
        ds = getattr(ex, "dataset", "unknown")
        dist[ds] = dist.get(ds, 0) + 1
    return dist

print(f"\n{'Split':<10} | {'#examples':>10} | {'#opinions':>10} | polarity dist")
print("-" * 80)
for name, exs in [("trainset", trainset), ("valset", valset), ("testset", testset)]:
    pdist, total_op = _polarity_dist(exs)
    ddist           = _dataset_dist(exs)
    print(f"{name:<10} | {len(exs):>10} | {total_op:>10} | {pdist}")
    print(f"{'':10}   {'dataset mix:':>12}  {ddist}")

## ✍️ DSPy Signatures — Seed Instructions for GEPA

Docstring của `Signature` chính là **thứ GEPA evolve** thông qua reflection LM
(xem [docs](https://dspy.ai/api/optimizers/GEPA/overview/)).

Theo quyết định: nạp **toàn bộ system prompt** (định nghĩa SOURCE/TARGET/POLAR_EXPRESSION/
POLARITY) làm docstring; user prompt **zero-shot** (chỉ gồm `text` + `sent_id`)
để instruction sinh ra generalize tốt và dùng được lại cho các technique khác.

In [ ]:
import dspy
from typing import Dict

class SSACoTSignature(dspy.Signature):
    # CHUỖI DOCSTRING NÀY CHÍNH LÀ INSTRUCTIONS MÀ GEPA SẼ TỐI ƯU
    __doc__ = f"""{SYSTEM_PROMPTS[DATASET_LANG[DATASETS[0]]].strip()}"""

    text = dspy.InputField(desc="The sentence or comment to be analyzed.")
    # sent_id = dspy.InputField(desc="The unique identifier for the sentence.")

    output_json = dspy.OutputField(desc="A valid JSON string containing the extraction results.")

print("=" * 80)
print(f"Initial signature instructions length = {len(SSACoTSignature.instructions)} chars")
print("=" * 80)
print(SSACoTSignature.instructions[:1500])
if len(SSACoTSignature.instructions) > 1500:
    print("...")

## 🧩 DSPy Modules

`dspy.ChainOfThought(SSACoTSignature)` tự động chèn predictor sinh **reasoning** trước
**output_json**, đúng tinh thần Chain-of-Thought.

Khi GEPA `compile()`, nó evolve `module.predict.predict.signature.instructions`
(tức docstring đã set ở Cell 8).

In [ ]:
class CoTSSAModule(dspy.Module):
    """Chain-of-Thought wrapper cho task SSA tiếng Việt.

    `forward(text, sent_id)` → object có `.output_json` (chuỗi JSON) và `.reasoning`
    (do `ChainOfThought` tự thêm). Metric sẽ parse output_json qua
    `extract_json_from_response` + `postprocess_response` để chuẩn hoá về SemEval format.
    """

    def __init__(self):
        super().__init__()
        self.predict = dspy.ChainOfThought(SSACoTSignature)

    def forward(self, text: str):
        return self.predict(text=text)

In [ ]:
# Structural smoke test (chưa cần LM — forward() sẽ test trong baseline eval ở Cell 21)
module = CoTSSAModule()

inner_sig = module.predict.predict.signature
assert "Polar_expression" in inner_sig.instructions, "System prompt không vào docstring!"
assert "text" in inner_sig.input_fields, "Missing 'text' input field"
# assert "sent_id" in inner_sig.input_fields, "Missing 'sent_id' input field"
assert "output_json" in inner_sig.output_fields, "Missing 'output_json' output field"
# ChainOfThought tự thêm 'reasoning' field
assert "reasoning" in inner_sig.output_fields, "ChainOfThought không thêm 'reasoning'"

print("Module OK ✓")
print(f"  Predictor type     = {module.predict.__class__.__name__}")
print(f"  Instructions chars = {len(inner_sig.instructions)}")
print(f"  Input fields       = {list(inner_sig.input_fields.keys())}")
print(f"  Output fields      = {list(inner_sig.output_fields.keys())}")

## 📊 Metric & Feedback Functions

GEPA's metric protocol: `(gold, pred, trace, pred_name, pred_trace) -> float | dspy.Prediction`
([docs](https://dspy.ai/api/optimizers/GEPA/overview/#implementing-feedback-metrics)).

- Khi `pred_name is None` (dùng cho `dspy.Evaluate` baseline) → trả `float`.
- Khi có `pred_name` (GEPA gọi để xin feedback) → trả `dspy.Prediction(score, feedback)`.

**Score = SF1 (Sentiment Graph F1)** — chính metric của SemEval-2022.

**Feedback** decompose 6 sub-metrics + liệt kê *missed/spurious tuples* + show *gold opinions*
để reflection LM có ground truth khi đề xuất prompt mới (theo recipe "decompose outcomes" +
"ground in checks" trong GEPA paper).

In [ ]:
def _compute_length_penalty(instr_chars: int) -> float:
    """
    Return multiplier ∈ [MIN_MULTIPLIER, 1.0]

    Zones
    -----
    [0, BASELINE]
        1.00

    [BASELINE, TARGET]
        1.00 -> 0.90

    [TARGET, HARD_CAP]
        0.90 -> 0.80

    > HARD_CAP
        Quadratic penalty:
        0.80 -> MIN_MULTIPLIER
    """

    # Zone 1
    if instr_chars <= PROMPT_LENGTH_BASELINE:
        return 1.0

    # Zone 2
    if instr_chars <= PROMPT_LENGTH_TARGET:
        ratio = (
            (instr_chars - PROMPT_LENGTH_BASELINE)
            / (PROMPT_LENGTH_TARGET - PROMPT_LENGTH_BASELINE)
        )
        return 1.0 - ratio * 0.10

    # Zone 3
    if instr_chars <= PROMPT_LENGTH_HARD_CAP:
        ratio = (
            (instr_chars - PROMPT_LENGTH_TARGET)
            / (PROMPT_LENGTH_HARD_CAP - PROMPT_LENGTH_TARGET)
        )
        return 0.90 - ratio * 0.10

    # Zone 4
    overflow = instr_chars - PROMPT_LENGTH_HARD_CAP

    quadratic_penalty = (overflow / OVERFLOW_SCALE) ** 2

    multiplier = (
        (1.0 - MAX_LENGTH_PENALTY)
        - 0.20 * quadratic_penalty
    )

    return max(MIN_MULTIPLIER, multiplier)

In [ ]:
def _parse_pred_to_sample(pred_obj, gold_text: str, gold_sent_id: str) -> dict:
    """Parse `pred.output_json` → SemEval sample format `{sent_id, text, opinions}`.

    Reuses `extract_json_from_response` + `postprocess_response` từ src.utils.postprocessing
    (tự fix các lỗi JSON phổ biến: backslash dư, trailing quote, unicode quote, etc.).
    """
    raw = getattr(pred_obj, "output_json", None) or "{}"
    if not isinstance(raw, str):
        raw = str(raw)
    extracted = extract_json_from_response(raw)
    normalized_str = postprocess_response(extracted, gold_text, gold_sent_id)
    try:
        return json.loads(normalized_str)
    except json.JSONDecodeError:
        return {"sent_id": gold_sent_id, "text": gold_text, "opinions": []}


def _normalize_metric_keys(metrics: Dict[str, float]) -> Dict[str, float]:
    """Map `calculate_all_metrics` keys ("Holder F1" → "Holder_F1") cho consistency."""
    return {k.replace(" ", "_"): float(v) for k, v in metrics.items()}


def _compute_per_doc_scores(gold_sample: dict, pred_sample: dict) -> Dict[str, Any]:
    """Tính 6 SemEval-2022 F1 cho 1 sample (per-doc, không phải corpus-level).

    Reuses `calculate_all_metrics` từ `semeval22_structured_sentiment/evaluation/evaluate.py`
    bằng cách gọi với dict 1 phần tử (cùng key cho gold + pred).

    Returns dict gồm 6 F1 (SF1, NSF1, Holder_F1, Target_F1, Exp_F1, Targeted_F1)
    + `_meta` (missed/spurious tuples cho feedback).
    """
    gt = convert_opinion_to_tuple(gold_sample)
    pt = convert_opinion_to_tuple(pred_sample)
    sid = str(gold_sample["sent_id"])

    # Single source of truth — same path as evaluate_single_dataset.py
    out: Dict[str, Any] = _normalize_metric_keys(
        calculate_all_metrics({sid: gt}, {sid: pt})
    )

    # Diagnostic: missed (FN) / spurious (FP) tuples — không có trong calculate_all_metrics
    missed_gold = [
        g for g in gt
        if not sent_tuples_in_list(g, pt, keep_polarity=True, mode="all")
    ]
    spurious_pred = [
        p for p in pt
        if not sent_tuples_in_list(p, gt, keep_polarity=True, mode="all")
    ]

    out["_meta"] = {
        "n_gold":        len(gt),
        "n_pred":        len(pt),
        "n_matched":     len(gt) - len(missed_gold),
        "missed_gold":   missed_gold,
        "spurious_pred": spurious_pred,
    }
    return out

In [ ]:
def _format_tuple_short(t) -> str:
    """Display 1 SemEval tuple compactly: (holder_idxs, target_idxs, exp_idxs, polarity)."""
    holder, target, exp, pol = t
    return f"(holder={sorted(holder)}, target={sorted(target)}, expr={sorted(exp)}, pol={pol})"


def _build_feedback(scores: Dict[str, Any], gold_sample: dict, pred_sample: dict) -> str:
    """Compose textual feedback decomposed by SemEval-2022 components.

    [UNRESTRICTED VERSION - High Rate Limit]
    Includes: Full original text, all metrics, exhaustive missed/spurious tuples,
    full gold opinions, full predicted opinions, and targeted hints.
    """
    sf1   = scores["SF1"]
    nsf1  = scores["NSF1"]
    h_f1  = scores["Holder_F1"]
    t_f1  = scores["Target_F1"]
    e_f1  = scores["Exp_F1"]
    tg_f1 = scores["Targeted_F1"]
    meta  = scores["_meta"]

    lines = [
        "==================================================",
        "EVALUATION REPORT FOR CURRENT INSTRUCTION",
        "==================================================",
        f"\n[ORIGINAL INPUT TEXT]",
        f"\"{gold_sample['text']}\"\n",
        "[METRICS SUMMARY]",
        f"SemEval-2022 SF1 = {sf1:.3f}",
        f"Component F1 — Holder={h_f1:.3f}, Target={t_f1:.3f}, Expression={e_f1:.3f}",
        f"Strict F1 — Targeted={tg_f1:.3f}, NSF1 (no polarity)={nsf1:.3f}",
        f"Tuples Count — Gold={meta['n_gold']}, Predicted={meta['n_pred']}, Matched={meta['n_matched']}.\n"
    ]

    # 1. LIỆT KÊ TOÀN BỘ LỖI SAI (KHÔNG GIỚI HẠN)
    if meta["missed_gold"]:
        lines.append("[FALSE NEGATIVES - You missed these entire tuples or extracted them incorrectly]:")
        # In ra tất cả, không dùng [:5] nữa
        lines.extend(f"  - {_format_tuple_short(t)}" for t in meta["missed_gold"])

    if meta["spurious_pred"]:
        lines.append("\n[FALSE POSITIVES - You hallucinated these or extracted boundaries wrongly]:")
        # In ra tất cả
        lines.extend(f"  - {_format_tuple_short(t)}" for t in meta["spurious_pred"])

    # 2. CUNG CẤP CẢ BẢN CHUẨN LẪN BẢN LỖI (Full JSON)
    lines.append("\n[RAW DATA COMPARISON]")

    gold_json = json.dumps(gold_sample.get("opinions", []), ensure_ascii=False, indent=2)
    lines.append("Gold Label (The correct answer):")
    lines.append(gold_json)

    # Thêm bản Dự đoán để Teacher LM so sánh trực tiếp
    pred_json = json.dumps(pred_sample.get("opinions", []), ensure_ascii=False, indent=2)
    lines.append("\nYour Prediction (The flawed output):")
    lines.append(pred_json)

    # 3. BẮT BỆNH VÀ KÊ ĐƠN (Diagnostic Hints)
    lines.append("\n[DIAGNOSTIC HINTS FOR SYSTEM PROMPT IMPROVEMENT]")

    if meta["n_pred"] == 0 and meta["n_gold"] > 0:
        lines.append(
            "- CRITICAL ERROR (ZERO EXTRACTION): You failed to extract any opinions, but the text contains sentiment. "
            "Update instructions to strictly enforce extraction of ANY sentiment, "
            "including teencode, slang, emojis, or subtle sarcasm."
        )

    if meta["n_pred"] > 0 and sf1 == 0.0:
        lines.append(
            "- CHAIN-BREAK ERROR: SF1 is 0.0 despite having predictions. "
            "SemEval SF1 requires Holder, Target, AND Expression to overlap simultaneously with the Gold label. "
            "One or more of your predicted components is completely off. Fix the weakest component."
        )

    if h_f1 < 0.3 and meta["n_gold"] > 0:
        lines.append(
            f"- HOLDER HALLUCINATION: Holder_F1 is critically low ({h_f1:.3f}). "
            "In Vietnamese, the sentiment holder is often omitted (pro-drop). "
            "Instruct the model: 'If the subject is not explicitly stated in the text, leave Source/Holder EMPTY. DO NOT infer or invent pronouns like tôi, khách hàng'."
        )

    component_f1 = {"Holder": h_f1, "Target": t_f1, "Expression": e_f1}
    weakest_name = min(component_f1, key=lambda k: component_f1[k])
    weakest_score = component_f1[weakest_name]

    if 0.0 < weakest_score < 0.7:
        lines.append(
            f"- BOUNDARY ERROR: '{weakest_name}' span detection is weak ({weakest_score:.3f}). "
            "Update instructions: 'Spans MUST be EXACT surface substrings of the input text. "
            "Do not include surrounding articles/prepositions (e.g., 'cái', 'sự', 'những'). Do not fix typos. Extract the minimal necessary phrase.'"
        )

    if t_f1 > 0.5 and tg_f1 < 0.2:
        lines.append(
            f"- TARGETED STRICT FAILURE: Target overlap is okay ({t_f1:.3f}) but strict match is poor ({tg_f1:.3f}). "
            "Targeted_F1 requires EXACT token match for the Target. "
            "Instruct the model to be laser-focused when defining the boundaries of the evaluated object/aspect."
        )

    if nsf1 - sf1 >= 0.15:
        lines.append(
            f"- POLARITY MISMATCH: NSF1 ({nsf1:.3f}) >> SF1 ({sf1:.3f}). "
            "Spans are mostly correct, but the POLARITY is misclassified. "
            "Update instructions to better define Positive/Negative/Neutral in Vietnamese context "
            "(e.g., handling sarcasm, idiomatic expressions, or double negatives)."
        )

    return "\n".join(lines)

def _build_length_feedback(instr_chars: int, multiplier: float) -> str:
    """
    Generate prompt-length feedback for the reflection LM.

    Goal:
    - Encourage concise instructions.
    - Penalize prompt bloat.
    - Prevent rule accumulation across GEPA iterations.
    """

    approx_tokens = instr_chars // 4

    chars_to_target = max(
        0,
        instr_chars - PROMPT_LENGTH_TARGET,
    )

    overflow = max(
        0,
        instr_chars - PROMPT_LENGTH_HARD_CAP,
    )

    penalty_pct = round(
        (1.0 - multiplier) * 100,
        1,
    )

    lines = ["\n[PROMPT LENGTH ANALYSIS]"]

    # ──────────────────────────────────────────────
    # No penalty
    # ──────────────────────────────────────────────
    if multiplier >= 1.0:
        lines.append(
            f"✓ Instruction length: {instr_chars} chars "
            f"(~{approx_tokens} tokens). "
            "Within efficient range. No penalty applied."
        )
        return "\n".join(lines)

    # ──────────────────────────────────────────────
    # General info
    # ──────────────────────────────────────────────
    lines += [
        f"⚠ Instruction length: {instr_chars} chars (~{approx_tokens} tokens).",
        f"  Current score multiplier: {multiplier:.2f}",
        f"  Length penalty applied: -{penalty_pct}% of score.",
        "",
        "RESEARCH FINDING:",
        "Performance often saturates once sufficient instruction complexity",
        "has been reached. Beyond that point, adding more rules, examples,",
        "or formatting guidance usually yields diminishing returns.",
        "",
        "ACTION REQUIRED — Revise the instruction to be MORE CONCISE:",
    ]

    # ──────────────────────────────────────────────
    # Severity levels
    # ──────────────────────────────────────────────
    if overflow > 0:

        if overflow >= 1000:
            severity = "EMERGENCY"
        elif overflow >= 500:
            severity = "SEVERE"
        else:
            severity = "CRITICAL"

        lines += [
            f"  [{severity}] Instruction exceeds hard cap by {overflow} chars.",
            f"  Remove at least {chars_to_target} chars to reach target length.",
            "  Large reductions are preferred over minor edits.",
            "  Merge redundant rules and remove generic guidance.",
            "  Keep only instructions that address observed errors.",
        ]

    elif instr_chars > PROMPT_LENGTH_TARGET:

        lines += [
            f"  [HIGH] Instruction is too verbose.",
            f"  Target: under {PROMPT_LENGTH_TARGET} chars.",
            "  Consolidate similar bullet points.",
            "  Remove explanations the model likely already knows.",
        ]

    else:

        lines += [
            "  [MODERATE] Slightly above efficient baseline.",
            "  Minor trimming is recommended.",
        ]

    # ──────────────────────────────────────────────
    # Anti-bloat guidance
    # ──────────────────────────────────────────────
    lines += [
        "",
        f"  Target length: < {PROMPT_LENGTH_TARGET} chars "
        f"(~{PROMPT_LENGTH_TARGET // 4} tokens).",
        "",
        "REVISION RULE:",
        "  Every newly added instruction must justify its existence",
        "  by correcting a specific MISSED or SPURIOUS tuple shown above.",
        "  If a rule does not address a demonstrated failure mode, delete it.",
        "  Prefer replacing existing rules over adding new rules.",
        "",
        "  Principle: Use the MINIMUM instruction necessary for the model",
        "  to perform the task correctly.",
    ]

    return "\n".join(lines)

In [ ]:
def ssa_metric_with_feedback(
    gold: dspy.Example,
    pred: dspy.Prediction,
    trace=None,
    pred_name: Optional[str] = None,
    pred_trace=None,
):
    # ── Lấy instruction length của candidate đang được evaluate ──
    try:
        current_instr = module.predict.predict.signature.instructions
        instr_chars   = len(current_instr)
    except Exception:
        instr_chars = 0

    length_multiplier = _compute_length_penalty(instr_chars)

    # ── Parse prediction ─────────────────────────────────────────
    gold_sample = {
        "sent_id":  str(gold.sent_id),
        "text":     gold.text,
        "opinions": gold.opinions,
    }

    try:
        pred_sample = _parse_pred_to_sample(pred, gold.text, str(gold.sent_id))
    except Exception as e:
        if pred_name is not None:
            return dspy.Prediction(
                score=0.0,
                feedback=(
                    f"Output failed to parse as JSON ({type(e).__name__}: {e}). "
                    "You MUST return a valid JSON object matching the exact schema. "
                    "No prose outside the JSON. Spans must be exact substrings of the input text."
                    + _build_length_feedback(instr_chars, length_multiplier)
                ),
            )
        elif trace is not None:
            return False
        else:
            return 0.0

    try:
        scores = _compute_per_doc_scores(gold_sample, pred_sample)
    except Exception as e:
        if pred_name is not None:
            return dspy.Prediction(
                score=0.0,
                feedback=(
                    f"JSON parsed but semantic structure is invalid: {str(e)}."
                    + _build_length_feedback(instr_chars, length_multiplier)
                )
            )
        elif trace is not None:
            return False
        else:
            return 0.0

    # ── Tính penalized score ──────────────────────────────────────
    raw_score       = float(scores["SF1"])
    penalized_score = raw_score * length_multiplier

    if pred_name is not None:
        feedback = (
            _build_feedback(scores, gold_sample, pred_sample)
            + _build_length_feedback(instr_chars, length_multiplier)
        )
        return dspy.Prediction(score=penalized_score, feedback=feedback)

    if trace is not None:
        return raw_score >= 1.0

    return penalized_score

In [ ]:
# Verify protocol with fake predictions (không cần LM)
class _FakePred:
    def __init__(self, output_json: str):
        self.output_json = output_json


_g = trainset[1]

# (1) Perfect prediction = gold
_perfect_json = json.dumps({
    "sent_id":  _g.sent_id,
    "text":     _g.text,
    "opinions": _g.opinions,
}, ensure_ascii=False)
_p_good = _FakePred(_perfect_json)

# (2) Empty prediction
_p_empty = _FakePred('{"opinions": []}')

# (3) Garbage
_p_bad = _FakePred("not json at all { { {")

# Test 1: pred_name=None → float
s1 = ssa_metric_with_feedback(_g, _p_good)
assert isinstance(s1, float), f"Expected float, got {type(s1)}"
print(f"[float, perfect-pred ]  SF1 = {s1:.3f}  (expected ~ 1.0)")

s2 = ssa_metric_with_feedback(_g, _p_empty)
print(f"[float, empty-pred   ]  SF1 = {s2:.3f}  (expected 0.0)")

# Test 2: pred_name=str → dspy.Prediction
out_p = ssa_metric_with_feedback(_g, _p_empty, pred_name="predict.predict")
assert isinstance(out_p, dspy.Prediction)
assert hasattr(out_p, "score") and hasattr(out_p, "feedback")
print(f"\n[Prediction, empty   ]  score = {out_p.score:.3f}")
print("Feedback preview (first 500 chars):")
print("-" * 80)
print(out_p.feedback)
print("-" * 80)

# Test 3: bad JSON
out_b = ssa_metric_with_feedback(_g, _p_bad, pred_name="predict.predict")
print(f"\n[Prediction, bad json]  score = {out_b.score:.3f}")
print(f"Feedback (first 200): {out_b.feedback}")

## 🤖 Model Setup

- **vLLM (student server)**: cell ngay dưới — khởi chạy API cục bộ; cần GPU + `vllm` trong môi trường kernel.
- **Student LM**: `dspy.LM` trỏ tới `STUDENT_BASE_URL` (OpenAI-compatible) — inference train/val/test.
- **Reflection LM**: model đề xuất prompt mới — nên mạnh (paper khuyến nghị GPT-5/Opus, `temperature=1.0`).

In [ ]:
# Khởi chạy vLLM OpenAI server cục bộ cho student LM (chạy sau cell Configuration).
# Tùy VRAM có thể thêm vào _vllm_cmd: --max-model-len 4096 --gpu-memory-utilization 0.9 --dtype bfloat16
import subprocess
import sys
import time
import urllib.error
import urllib.request

if "_vllm_proc" in globals() and _vllm_proc is not None and _vllm_proc.poll() is None:
    _vllm_proc.terminate()
    try:
        _vllm_proc.wait(timeout=15)
    except subprocess.TimeoutExpired:
        _vllm_proc.kill()

_vllm_env = os.environ.copy()
_hf = (
    _vllm_env.get("HUGGINGFACE_API_KEY")
    or _vllm_env.get("HF_TOKEN")
    or _vllm_env.get("HUGGINGFACE_HUB_TOKEN")
    or _vllm_env.get("HUGGINGFACE_API", "")
)
if _hf:
    _vllm_env.setdefault("HF_TOKEN", _hf)
    _vllm_env.setdefault("HUGGINGFACE_HUB_TOKEN", _hf)

_vllm_cmd = [
    sys.executable,
    "-m",
    "vllm.entrypoints.openai.api_server",
    "--model",
    VLLM_MODEL_ID,
    "--host",
    "127.0.0.1",
    "--port",
    str(VLLM_PORT),
    "--dtype",
    "bfloat16"
]

_vllm_log = OUTPUT_DIR / "vllm_server.log"
_flog = open(_vllm_log, "a", encoding="utf-8", buffering=1)
_flog.write(f"\n\n==== vLLM start {time.strftime('%Y-%m-%d %H:%M:%S')} ====\n")
_flog.write(" ".join(_vllm_cmd) + "\n")
_flog.flush()

print("Starting vLLM — log:", _vllm_log)
_vllm_proc = subprocess.Popen(
    _vllm_cmd,
    env=_vllm_env,
    stdout=_flog,
    stderr=subprocess.STDOUT,
    cwd=str(PROJECT_ROOT),
)

_health = STUDENT_BASE_URL.rstrip("/") + "/models"
_deadline = time.time() + float(os.getenv("VLLM_START_TIMEOUT_S", "1200"))
_last_err = None
while time.time() < _deadline:
    if _vllm_proc.poll() is not None:
        _flog.close()
        tail = _vllm_log.read_text(encoding="utf-8", errors="replace")[-4000:]
        raise RuntimeError(
            f"vLLM exited early (code={_vllm_proc.returncode}). See {_vllm_log}. Tail:\n{tail}"
        )
    try:
        with urllib.request.urlopen(_health, timeout=5) as r:
            if r.status == 200:
                print(f"vLLM ready: {_health}")
                break
    except (urllib.error.URLError, TimeoutError, OSError) as e:
        _last_err = e
    time.sleep(2.0)
else:
    if _vllm_proc.poll() is None:
        _vllm_proc.terminate()
    _flog.close()
    raise RuntimeError(f"vLLM did not become ready in time. Last error: {_last_err}. Log: {_vllm_log}")

In [ ]:
student_lm = dspy.LM(
    STUDENT_MODEL,
    max_tokens=STUDENT_MAX_TOKENS,
    temperature=0.0,
    api_base=STUDENT_BASE_URL,
    api_key=STUDENT_API_KEY,
    cache=True,
)
dspy.configure(lm=student_lm, track_usage=True)
print(f"Student LM configured: {STUDENT_MODEL} @ {STUDENT_BASE_URL}")

In [ ]:
reflection_lm = dspy.LM(
    model=REFLECTION_MODEL,
    api_key=GEMINI_API_KEY,
    temperature=1.0,
    max_tokens=REFLECTION_MAX_TOKENS,
    max_retries=5,
    timeout=180,
)

print(f"Reflection LM configured: {REFLECTION_MODEL}")

## 🚀 GEPA Optimization

- `auto={light, medium, heavy}` quyết định **rollout budget** (paper khuyến nghị `medium`/`heavy` cho production).
- `log_dir`: lưu state để **resume** nếu interrupt giữa chừng.
- `track_stats=True`: lưu `detailed_results` (Pareto front, val aggregate scores) cho phần Inspect.
- `reflection_lm`: model mạnh đề xuất prompt mới (cấu hình trong section **Model Setup**).

In [ ]:
optimizer = dspy.GEPA(
    metric=ssa_metric_with_feedback,
    auto=AUTO_BUDGET,
    reflection_minibatch_size=REFLECTION_MINIBATCH_SIZE,
    max_metric_calls=MAX_METRIC_CALLS,
    max_full_evals=MAX_FULL_EVALS,
    candidate_selection_strategy="pareto",
    component_selector="all",
    use_merge=True,
    max_merge_invocations=5,
    skip_perfect_score=True,
    num_threads=NUM_THREADS,
    track_stats=True,
    log_dir=str(GEPA_LOG_DIR),
    reflection_lm=reflection_lm,
    seed=SEED,
)

print(f"Starting GEPA: auto={AUTO_BUDGET}, "
      f"trainset={len(trainset)}, valset={len(valset)}, "
      f"reflection_lm={REFLECTION_MODEL}")
print(f"Logs: {GEPA_LOG_DIR}\n")

optimized_module = optimizer.compile(
    student=module,
    trainset=trainset,
    valset=valset,
)

print("\n✓ GEPA compile finished.")

In [ ]:
save_path = OUTPUT_DIR / "optimized_cot.json"
optimized_module.save(str(save_path))
print(f"Saved optimized module to: {save_path}")

opt_instr = optimized_module.predict.predict.signature.instructions
print(f"\nOptimized instruction length = {len(opt_instr)} chars")
print("=" * 80)
print(opt_instr)

## 🔍 Inspect Optimized Prompts

So sánh **baseline instruction** (system prompt gốc) vs **optimized instruction**
(GEPA evolved). Đồng thời in `detailed_results` (Pareto front, total metric calls)
để hiểu chi phí và quá trình tối ưu.

In [ ]:
base_instr = SSACoTSignature.instructions
opt_instr  = optimized_module.predict.predict.signature.instructions

print("=" * 80)
print("BASELINE instruction (truncated to 1500):")
print("=" * 80)
print(base_instr[:1500])
print("\n" + "=" * 80)
print("OPTIMIZED instruction (truncated to 1500):")
print("=" * 80)
print(opt_instr[:1500])

# Length diff
print("\n" + "-" * 80)
print(f"Baseline length  = {len(base_instr)} chars, {len(base_instr.split())} words")
print(f"Optimized length = {len(opt_instr)} chars, {len(opt_instr.split())} words")

# detailed_results (chỉ có khi track_stats=True)
res = getattr(optimized_module, "detailed_results", None)
if res is not None:
    print("\n" + "=" * 80)
    print("GEPA detailed_results:")
    print("=" * 80)
    for attr in ("total_metric_calls", "num_full_val_evals"):
        val = getattr(res, attr, None)
        if val is not None:
            print(f"  {attr:<25s} = {val}")
    val_scores = getattr(res, "val_aggregate_scores", None)
    if val_scores is not None:
        try:
            print(f"  val_aggregate_scores      = {[round(float(x), 4) for x in val_scores]}")
        except Exception:
            print(f"  val_aggregate_scores      = {val_scores}")
    log_dir = getattr(res, "log_dir", None)
    if log_dir:
        print(f"  log_dir                   = {log_dir}")
else:
    print("\n(No detailed_results — track_stats=False?)")